# 🚗 Used Car Price Prediction using Decision Tree Regression
Real-world project using the CarDekho dataset.

**Dataset:** https://raw.githubusercontent.com/amankharwal/Website-data/master/car%20data.csv

> Run the first cell to download the dataset automatically.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

url="https://raw.githubusercontent.com/amankharwal/Website-data/master/car%20data.csv"
df=pd.read_csv(url)
df.head()

## Dataset Overview

In [ ]:
print(df.shape)
print(df.info())
print(df.describe(include='all'))
print("\nMissing Values\n",df.isnull().sum())
print("\nDuplicates:",df.duplicated().sum())

## Feature Engineering

In [ ]:
import datetime
current_year=datetime.datetime.now().year
df['Car_Age']=current_year-df['Year']
df=df.drop('Year',axis=1)
df.head()

## Exploratory Data Analysis

In [ ]:
df.hist(figsize=(12,8))
plt.tight_layout()
plt.show()

numeric=df.select_dtypes(include=np.number)
plt.figure(figsize=(8,6))
plt.imshow(numeric.corr(),cmap='coolwarm')
plt.xticks(range(len(numeric.columns)),numeric.columns,rotation=90)
plt.yticks(range(len(numeric.columns)),numeric.columns)
plt.colorbar()
plt.title("Correlation Heatmap")
plt.show()

## Prepare Features

In [ ]:
X=df.drop('Selling_Price',axis=1)
y=df['Selling_Price']

cat=X.select_dtypes(include='object').columns
num=X.select_dtypes(exclude='object').columns

preprocessor=ColumnTransformer([
('cat',OneHotEncoder(handle_unknown='ignore'),cat),
('num','passthrough',num)
])

X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,random_state=42)

## Baseline Decision Tree

In [ ]:
baseline=Pipeline([
('prep',preprocessor),
('model',DecisionTreeRegressor(random_state=42))
])

baseline.fit(X_train,y_train)

pred=baseline.predict(X_test)

print("MAE:",mean_absolute_error(y_test,pred))
print("RMSE:",mean_squared_error(y_test,pred)**0.5)
print("R2:",r2_score(y_test,pred))
print("Train R2:",baseline.score(X_train,y_train))
print("Test R2:",baseline.score(X_test,y_test))

## Hyperparameter Tuning using GridSearchCV

In [ ]:
pipe=Pipeline([
('prep',preprocessor),
('model',DecisionTreeRegressor(random_state=42))
])

params={
'model__criterion':['squared_error','friedman_mse'],
'model__max_depth':[3,5,7,10,None],
'model__min_samples_split':[2,5,10,20],
'model__min_samples_leaf':[1,2,5,10],
'model__max_leaf_nodes':[10,20,40,None],
'model__ccp_alpha':[0.0,0.001,0.01]
}

grid=GridSearchCV(pipe,
params,
cv=5,
scoring='r2',
n_jobs=-1)

grid.fit(X_train,y_train)

print("Best Parameters:")
print(grid.best_params_)
print("Best CV Score:",grid.best_score_)

## Evaluate Best Model

In [ ]:
best=grid.best_estimator_

pred=best.predict(X_test)

mae=mean_absolute_error(y_test,pred)
mse=mean_squared_error(y_test,pred)
rmse=np.sqrt(mse)
r2=r2_score(y_test,pred)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R2:",r2)
print("Train R2:",best.score(X_train,y_train))
print("Test R2:",best.score(X_test,y_test))

## Feature Importance

In [ ]:
encoder=best.named_steps['prep']
feature_names=encoder.get_feature_names_out()
importance=best.named_steps['model'].feature_importances_

imp=(pd.DataFrame({'Feature':feature_names,'Importance':importance})
      .sort_values('Importance',ascending=False)
      .head(15))

print(imp)

plt.figure(figsize=(8,6))
plt.barh(imp['Feature'][::-1],imp['Importance'][::-1])
plt.title("Top 15 Feature Importances")
plt.show()

## Visualize Decision Tree

In [ ]:
plt.figure(figsize=(22,10))
plot_tree(best.named_steps['model'],
filled=True,
rounded=True,
max_depth=3,
fontsize=8)
plt.show()

## Predict Price of a New Car

In [ ]:
sample=pd.DataFrame({
'Present_Price':[8.5],
'Kms_Driven':[35000],
'Fuel_Type':['Petrol'],
'Seller_Type':['Dealer'],
'Transmission':['Manual'],
'Owner':[0],
'Car_Age':[4]
})

print("Predicted Selling Price (Lakhs):",best.predict(sample)[0])

## Interpretation of Results
- Compare Train R² and Test R² to detect overfitting.
- Higher R² and lower RMSE indicate better performance.
- GridSearchCV selects the best hyperparameters using 5-fold cross-validation.
- Feature importance identifies the variables that most influence selling price.
- If Train R² is much higher than Test R², the model is overfitting; otherwise, it generalizes well.

## Conclusion
This notebook demonstrates a complete real-world Decision Tree Regression workflow: data loading, preprocessing, EDA, model building, hyperparameter tuning, evaluation, interpretation, feature importance, visualization, and prediction.